# Clase 23: Redes Neuronales — De la Línea Recta a la Curva

**Objetivo:** Entender qué es una red neuronal, cómo se conecta con la regresión logística que ya conocemos, y entrenar nuestro primer MLP en sklearn.

**Flujo de la clase:**
1. El problema: logística falla en datos curvos
2. De logística a neurona: extendiendo lo que ya saben
3. Conectar neuronas: capas, conexiones, qué detecta cada una
4. MLPClassifier paso a paso: import, parámetros, código completo
5. Experimentos con learning rate y arquitectura
6. MLP en MNIST: comparación con logística y RF
7. TensorFlow Playground: visualizar redes neuronales en acción
8. Limitaciones del MLP y qué viene después

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import plotly.express as px
import plotly.graph_objects as go
import time
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.inspection import DecisionBoundaryDisplay

### Función auxiliar para graficar fronteras de decisión

La usaremos muchas veces — grafica los puntos y la frontera que el modelo aprendió.

In [ ]:
def plot_frontera(modelo, X, y, titulo="", ax=None, scaler=None):
    """Grafica la frontera de decisión de un modelo 2D."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4.5))
    
    # Crear grid
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    
    if scaler is not None:
        grid_scaled = scaler.transform(grid)
        Z = modelo.predict(grid_scaled)
    else:
        Z = modelo.predict(grid)
    Z = Z.reshape(xx.shape)
    
    # Colores suaves para regiones
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="RdBu")
    ax.contour(xx, yy, Z, colors="gray", linewidths=0.5)
    
    # Puntos
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap="RdBu",
                        edgecolors="white", s=20, linewidths=0.5)
    acc = accuracy_score(y, modelo.predict(scaler.transform(X) if scaler else X))
    ax.set_title(f"{titulo}\nAccuracy: {acc:.1%}", fontweight="bold", fontsize=11)
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
    return ax

---
## 1. El Problema: La Logística Falla en Datos Curvos

Empecemos generando datos que **no son linealmente separables** y veamos qué pasa con la logística.

In [ ]:
# Generar dos datasets no lineales
X_moons, y_moons = make_moons(n_samples=1000, noise=0.25, random_state=42)
X_circles, y_circles = make_circles(n_samples=1000, noise=0.1, factor=0.4, random_state=42)

print("Medialunas:", X_moons.shape, "— 2 grupos con forma de C entrelazados")
print("Círculos:", X_circles.shape, "— un grupo dentro de otro")

In [ ]:
# Entrenar logística en ambos
lr_moons = LogisticRegression().fit(X_moons, y_moons)
lr_circles = LogisticRegression().fit(X_circles, y_circles)

# Entrenar RF en ambos
rf_moons = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_moons, y_moons)
rf_circles = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_circles, y_circles)

# Visualizar las 4 fronteras
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
plot_frontera(lr_moons, X_moons, y_moons, "Logística en Medialunas", axes[0, 0])
plot_frontera(rf_moons, X_moons, y_moons, "Random Forest en Medialunas", axes[0, 1])
plot_frontera(lr_circles, X_circles, y_circles, "Logística en Círculos", axes[1, 0])
plot_frontera(rf_circles, X_circles, y_circles, "Random Forest en Círculos", axes[1, 1])
plt.tight_layout()
plt.show()

### Interpretación

- **Logística en medialunas (~84%):** traza una línea recta. No puede seguir la curva de las medialunas. No importa cuántas épocas entrenes — una recta es una recta.
- **Logística en círculos (~50%):** es literalmente azar. No existe ninguna línea recta que separe un círculo interno de uno externo.
- **RF en medialunas (~100%):** funciona, pero la frontera es **dentada** — cortes rectangulares, no una curva suave.
- **RF en círculos (~99%):** también funciona, pero la frontera parece un polígono, no un círculo.

**Pregunta:** ¿existe un modelo que trace **curvas suaves** automáticamente?

---
## 2. De la Logística a la Neurona

La respuesta es **sí** — las redes neuronales. Y resulta que ya conocen la pieza fundamental: la regresión logística.

### La logística ES una neurona

Recuerden qué hace la logística:
1. Toma las variables ($x_1, x_2, \ldots$) y las multiplica por coeficientes ($\beta_1, \beta_2, \ldots$)
2. Suma todo + un intercepto: $z = \beta_1 x_1 + \beta_2 x_2 + \ldots + \beta_0$
3. Pasa $z$ por la función sigmoid: $\sigma(z) = \frac{1}{1+e^{-z}}$

Esos 3 pasos son **exactamente** lo que hace una neurona artificial:

| Logística | Neurona | Significado |
|---|---|---|
| Coeficientes β | Pesos w | Cuánto influye cada variable |
| Intercept β₀ | Bias b | Desplaza la frontera |
| Sigmoid σ | Activación f | Transforma la suma en salida |

**No es un concepto nuevo.** Es el mismo concepto de la logística, repetido muchas veces y conectado en capas.

In [ ]:
# Veamos los coeficientes de la logística en medialunas
print("=== Logística en medialunas ===")
print(f"Coeficientes (β / pesos w): {lr_moons.coef_[0].round(3)}")
print(f"Intercept (β₀ / bias b):    {lr_moons.intercept_[0]:.3f}")
print()
print("La neurona hace exactamente esto:")
print(f"z = {lr_moons.coef_[0][0]:.3f}·x₁ + {lr_moons.coef_[0][1]:.3f}·x₂ + ({lr_moons.intercept_[0]:.3f})")
print(f"salida = sigmoid(z) → probabilidad entre 0 y 1")
print()
print("El problema: z = w₁x₁ + w₂x₂ + b es la ecuación de una LÍNEA RECTA.")
print("Una neurona sola = una línea. Para curvas, necesitamos MUCHAS neuronas.")

### ¿Qué cambia en una RED neuronal?

En vez de **1 neurona** (= logística = 1 línea recta), ponemos **muchas neuronas en capas**:
- Cada neurona traza su propia línea con sus propios pesos
- La siguiente capa **combina** esas líneas
- La combinación de muchas líneas dobladas puede formar una **curva**

Además, en las capas internas se usa una función de **activación no lineal** (como ReLU, Tanh, u otras):
- **ReLU**(z) = max(0, z) es la más popular hoy por su simplicidad, pero no es la única opción.
- Lo importante es que sea **no lineal** — sin eso, agregar más capas no agrega capacidad.

---
## 3. MLPClassifier: Nuestra Primera Red Neuronal

MLP = **M**ulti-**L**ayer **P**erceptron. Vive en `sklearn.neural_network`.

Tiene la **misma API** que LogisticRegression y RandomForest: `.fit()`, `.predict()`, `.predict_proba()`, `.score()`.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# IMPORTANTE: las redes neuronales NECESITAN escalar (igual que K-Means)
scaler_m = StandardScaler()
X_moons_s = scaler_m.fit_transform(X_moons)

# Crear el MLP
mlp_moons = MLPClassifier(
    hidden_layer_sizes=(32, 16),   # 2 capas ocultas: 32 y 16 neuronas
    activation="relu",             # ReLU en capas ocultas (default)
    max_iter=500,                  # Máximo de épocas
    learning_rate_init=0.001,      # Tamaño del paso (default)
    early_stopping=True,           # para si no mejora en validación
    random_state=42
)

mlp_moons.fit(X_moons_s, y_moons)

print(f"Accuracy: {mlp_moons.score(X_moons_s, y_moons):.1%}")
print(f"Épocas usadas: {mlp_moons.n_iter_} (de máximo {mlp_moons.max_iter})")
print(f"Capas: {mlp_moons.hidden_layer_sizes}")
print(f"Total de pesos: {sum(w.size for w in mlp_moons.coefs_)}")

In [ ]:
# Comparar las 3 fronteras: Logística vs RF vs MLP
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

plot_frontera(lr_moons, X_moons, y_moons, "Logística\n(1 línea recta)", axes[0])
plot_frontera(rf_moons, X_moons, y_moons, "Random Forest\n(cortes rectangulares)", axes[1])
plot_frontera(mlp_moons, X_moons, y_moons, "MLP (32, 16)\n(curva suave)", axes[2], scaler=scaler_m)

plt.suptitle("Logística vs RF vs Red Neuronal — mismos datos, distinta frontera",
             fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

### Interpretación

- **Logística:** línea recta. No puede separar las medialunas.
- **RF:** lo separa pero con cortes rectangulares — la frontera es dentada.
- **MLP:** traza una **curva suave** que sigue la forma natural de los datos.

La red neuronal aprendió automáticamente que los datos tienen forma de medialuna. No le dijimos nada — solo le dimos los puntos y las etiquetas.

In [ ]:
# Lo mismo con círculos
scaler_c = StandardScaler()
X_circles_s = scaler_c.fit_transform(X_circles)

mlp_circles = MLPClassifier(
    hidden_layer_sizes=(32, 16), activation="relu",
    max_iter=500, early_stopping=True, random_state=42)
mlp_circles.fit(X_circles_s, y_circles)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
plot_frontera(lr_circles, X_circles, y_circles, "Logística\n(imposible)", axes[0])
plot_frontera(rf_circles, X_circles, y_circles, "Random Forest\n(polígono)", axes[1])
plot_frontera(mlp_circles, X_circles, y_circles, "MLP (32, 16)\n(círculo)", axes[2], scaler=scaler_c)

plt.suptitle("Círculos concéntricos — la logística no puede, el MLP sí",
             fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. La Curva de Loss: El "Electrocardiograma" de la Red

Después de entrenar, **siempre** hay que verificar que la red convergió. La curva de loss muestra el error en cada época.

In [ ]:
fig = px.line(y=mlp_moons.loss_curve_,
    title="Curva de Loss — MLP en Medialunas",
    labels={"index": "Época", "value": "Loss (Cross-Entropy)"})
fig.update_layout(template="plotly_white")
fig.show()

### Interpretación de la curva de loss

- **Baja rápido al inicio:** la red aprende los patrones gruesos ("hay dos grupos").
- **Se estabiliza después:** ya aprendió lo principal, los ajustes son finos.
- **Si no baja:** la red es muy pequeña o el learning rate está mal.
- **Si oscila (sube y baja):** el learning rate es muy grande.

Esta curva se ve saludable: baja suavemente y se estabiliza. ✅

---
## 5. Experimentos con Learning Rate

El **learning rate** controla qué tan grandes son los pasos de ajuste de pesos. Es el hiperparámetro más importante de una red neuronal. Veamos qué pasa cuando lo cambiamos.

In [ ]:
learning_rates = {
    "LR = 0.1 (muy grande)": 0.1,
    "LR = 0.01": 0.01,
    "LR = 0.001 (default)": 0.001,
    "LR = 0.0001 (muy chico)": 0.0001,
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
loss_curves = {}

for i, (nombre, lr_val) in enumerate(learning_rates.items()):
    mlp_lr = MLPClassifier(
        hidden_layer_sizes=(32, 16), activation="relu",
        max_iter=300, learning_rate_init=lr_val,
        early_stopping=False,  # Desactivamos para ver el efecto completo
        random_state=42)
    mlp_lr.fit(X_moons_s, y_moons)
    loss_curves[nombre] = mlp_lr.loss_curve_
    
    plot_frontera(mlp_lr, X_moons, y_moons, nombre, axes[i], scaler=scaler_m)

plt.suptitle("Efecto del Learning Rate en la frontera de decisión",
             fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Comparar las curvas de loss
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#C82B40", "#EA580C", "#16A34A", "#2563EB"]
for (nombre, curva), color in zip(loss_curves.items(), colors):
    ax.plot(curva, label=nombre, color=color, linewidth=2)
ax.set_xlabel("Época", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.set_title("Curvas de Loss — Efecto del Learning Rate", fontweight="bold", fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.0)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Interpretación del Learning Rate

| Learning Rate | Curva de Loss | Frontera | Diagnóstico |
|---|---|---|---|
| **0.1 (muy grande)** | Oscila, no baja establemente | Inestable, puede fallar | Los pesos "saltan" demasiado, la red nunca se estabiliza |
| **0.01** | Baja rápido, se estabiliza | Buena curva | Funciona bien para este problema |
| **0.001 (default)** | Baja suavemente | Curva suave | El estándar. Casi siempre funciona |
| **0.0001 (muy chico)** | Baja muy lento | Puede no terminar de aprender | Los pasos son tan chicos que necesitaría muchas más épocas |

**Regla práctica:** empezar con **0.001** (default). Si la curva oscila → bajar. Si es muy lenta → subir.

---
## 6. Experimentos con la Arquitectura

¿Cuántas neuronas y capas necesitamos? Veamos cómo evoluciona la frontera.

In [ ]:
arquitecturas = {
    "1 neurona\n= logística": (1,),
    "4 neuronas\n(1 capa)": (4,),
    "16 neuronas\n(1 capa)": (16,),
    "2 capas\n(32, 16)": (32, 16),
    "3 capas\n(64, 32, 16)": (64, 32, 16),
}

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for i, (nombre, capas) in enumerate(arquitecturas.items()):
    mlp_arch = MLPClassifier(
        hidden_layer_sizes=capas, activation="relu",
        max_iter=500, early_stopping=True, random_state=42)
    mlp_arch.fit(X_moons_s, y_moons)
    
    n_params = sum(w.size for w in mlp_arch.coefs_) + sum(b.size for b in mlp_arch.intercepts_)
    plot_frontera(mlp_arch, X_moons, y_moons,
                 f"{nombre}\n({n_params} pesos)", axes[i], scaler=scaler_m)

plt.suptitle("Más neuronas = fronteras más complejas (pero cuidado con overfitting)",
             fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

### Interpretación

- **1 neurona:** es literalmente la logística — una línea recta. ~84%.
- **4 neuronas:** empieza a curvarse. Cada neurona traza una línea diferente, la combinación crea un "codo".
- **16 neuronas:** sigue bien la forma de las medialunas.
- **2 capas (32, 16):** curva más suave y precisa.
- **3 capas (64, 32, 16):** similar — para este problema simple, más capas no agregan mucho.

**La forma piramidal** (64 → 32 → 16) es la recomendación estándar: cada capa comprime la información gradualmente.

---
## 7. MLP en MNIST: Comparación con Logística y RF

Apliquemos la red neuronal al dataset de imágenes de la clase 22. ¿Mejora el 95% de Random Forest?

In [ ]:
from sklearn.datasets import fetch_openml

print("Cargando MNIST (puede tomar unos segundos)...")
X_full, y_full = fetch_openml("mnist_784", version=1,
                               return_X_y=True, as_frame=False, parser="auto")

# Subset de 15K para velocidad
rng = np.random.default_rng(42)
idx = rng.choice(len(X_full), 15000, replace=False)
X_mnist, y_mnist = X_full[idx], y_full[idx]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_mnist, y_mnist, test_size=0.2, random_state=42, stratify=y_mnist)

# Normalizar pixeles (obligatorio para MLP)
X_train_n = X_train / 255.0
X_test_n = X_test / 255.0

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Rango normalizado: {X_train_n.min():.0f} - {X_train_n.max():.0f}")

In [ ]:
# === Modelo 1: Logística ===
t0 = time.time()
lr_mnist = LogisticRegression(max_iter=1000, random_state=42)
lr_mnist.fit(X_train_n, y_train)
t_lr = time.time() - t0
acc_lr = accuracy_score(y_test, lr_mnist.predict(X_test_n))
print(f"Logística:     {acc_lr:.1%}  ({t_lr:.1f}s)")

# === Modelo 2: Random Forest ===
t0 = time.time()
rf_mnist = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_mnist.fit(X_train, y_train)  # RF no necesita normalizar
t_rf = time.time() - t0
acc_rf = accuracy_score(y_test, rf_mnist.predict(X_test))
print(f"Random Forest: {acc_rf:.1%}  ({t_rf:.1f}s)")

# === Modelo 3: MLP ===
t0 = time.time()
mlp_mnist = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    max_iter=50,
    early_stopping=True,
    random_state=42)
mlp_mnist.fit(X_train_n, y_train)
t_mlp = time.time() - t0
acc_mlp = accuracy_score(y_test, mlp_mnist.predict(X_test_n))
print(f"MLP (128, 64): {acc_mlp:.1%}  ({t_mlp:.1f}s)  [{mlp_mnist.n_iter_} épocas]")

In [ ]:
# Comparación visual
modelos = ["Logística", "Random Forest", "MLP (128, 64)"]
accs = [acc_lr, acc_rf, acc_mlp]
tiempos = [t_lr, t_rf, t_mlp]
colores = ["#2563EB", "#16A34A", "#C82B40"]

fig = go.Figure()
fig.add_trace(go.Bar(x=modelos, y=accs, marker_color=colores,
                     text=[f"{a:.1%}" for a in accs], textposition="outside"))
fig.update_layout(title="Comparación de modelos en MNIST (15K imágenes)",
                  yaxis_range=[0.85, 1.0], template="plotly_white",
                  yaxis_title="Accuracy")
fig.show()

In [ ]:
# Curva de loss del MLP en MNIST
fig = px.line(y=mlp_mnist.loss_curve_,
    title="Curva de Loss — MLP en MNIST",
    labels={"index": "Época", "value": "Loss"})
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
# Confusion matrix del MLP
y_pred_mlp = mlp_mnist.predict(X_test_n)
cm = confusion_matrix(y_test, y_pred_mlp, labels=[str(i) for i in range(10)])

fig, ax = plt.subplots(figsize=(8, 7))
import seaborn as sns
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds", ax=ax,
            xticklabels=range(10), yticklabels=range(10))
ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
ax.set_title(f"Confusion Matrix — MLP ({acc_mlp:.1%})", fontweight="bold")
plt.tight_layout()
plt.show()

### Interpretación

| Modelo | Accuracy | Tiempo | Tipo de frontera |
|---|---|---|---|
| Logística | ~90% | Rápido | Línea recta |
| Random Forest | ~95% | Medio | Cortes rectangulares |
| **MLP (128, 64)** | **~97%** | Lento | **Curva suave** |

- El MLP mejora especialmente en los dígitos difíciles (3↔8, 9↔4).
- Pero tarda más y es una "caja negra" — no podemos inspeccionar qué pixeles importan tan fácilmente.
- La curva de loss baja y se estabiliza: la red convergió correctamente.

---
## 8. Ejercicio: Experimentar con el MLP en MNIST

**⏱️ 10 minutos:**

1. Cambien la arquitectura a `(256, 128, 64)`. ¿Mejora el accuracy?
2. Grafiquen la curva de loss. ¿Convergió?
3. Prueben con `learning_rate_init=0.01`. ¿La curva oscila?
4. Prueben con `(16,)` (una sola capa chica). ¿Es suficiente para MNIST?
5. **Reflexión:** ¿más neuronas siempre es mejor? ¿Cuál es el costo?

In [ ]:
# Tu código aquí


---
## 9. TensorFlow Playground: Visualizar Redes Neuronales en Acción

### ¿Qué es el TensorFlow Playground?

Es una herramienta web interactiva de Google que permite **ver en tiempo real** cómo una red neuronal aprende. Pueden acceder en:

👉 **https://playground.tensorflow.org**

### ¿Qué puedes controlar?

| Elemento | Qué controlas | Equivalente en sklearn |
|---|---|---|
| **Dataset** (arriba izquierda) | Medialunas, círculos, XOR, espiral | `make_moons()`, `make_circles()` |
| **Features** (columna izquierda) | Qué inputs recibe la red: x₁, x₂, x₁², sin(x₁)... | Feature engineering manual |
| **Capas ocultas** (+/- botones) | Cuántas capas y neuronas por capa | `hidden_layer_sizes=(32, 16)` |
| **Learning rate** (arriba) | Tamaño del paso | `learning_rate_init=0.001` |
| **Activation** (arriba) | ReLU, Tanh, Sigmoid, Linear | `activation="relu"` |
| **Regularization** (arriba) | L1, L2, None | `alpha=0.0001` |

### Lo más importante: ¿Qué muestran los colores dentro de cada neurona?

Cada neurona de las capas ocultas muestra **lo que aprendió** — el patrón que detecta. Esto es clave:

- **Tú NO le dices a las neuronas qué detectar.** La red lo aprende sola a través de backpropagation.
- Los colores que ves dentro de cada neurona son el **resultado** del entrenamiento, no algo que tú configures.
- Lo que TÚ controlas son los **features de entrada** (columna izquierda): x₁, x₂, x₁², sin(x₁), etc. Eso es **feature engineering** — elegir qué datos le das a la red.

### Ejercicio guiado con el Playground

Abran **https://playground.tensorflow.org** y sigan estos pasos:

#### Experimento 1: Círculos con 1 neurona
1. Dataset: **Circle** (arriba izquierda)
2. Features: solo **x₁** y **x₂** (desmarcar los demás)
3. Capas: **1 capa con 1 neurona**
4. Click ▶ (Play). Observen: la frontera es una **línea recta**. No puede separar los círculos.

#### Experimento 2: Agregar neuronas
1. Misma configuración, pero ahora **1 capa con 4 neuronas**
2. Click ▶. Observen cómo cada neurona aprende un patrón diferente (los colores dentro de cada cuadrado).
3. La frontera empieza a curvarse.

#### Experimento 3: Agregar capas
1. **2 capas: 4 y 2 neuronas**
2. Click ▶. La frontera es más suave.
3. Observen: la primera capa detecta patrones simples, la segunda los **combina**.

#### Experimento 4: El poder del feature engineering
1. Vuelvan a **1 capa, 1 neurona**
2. Activen el feature **x₁²** y **x₂²**
3. Click ▶. ¡Ahora SÍ separa los círculos con 1 sola neurona!
4. **¿Por qué?** Porque x₁² + x₂² = radio². Al darle el radio como feature, el problema se vuelve linealmente separable.

#### Experimento 5: Learning rate
1. Dataset: **Spiral**, 2 capas de 6 neuronas
2. Prueben LR = 0.1, 0.01, 0.001
3. Observen cómo LR = 0.1 oscila y LR = 0.001 es lento.

### La lección clave del Playground

**Las neuronas aprenden solas.** Tú controlas la estructura (cuántas capas, cuántas neuronas, qué features de entrada), pero el **contenido** de cada neurona — qué patrón detecta — es algo que la red descubre por sí misma a través de backpropagation.

Es exactamente lo que vimos en código: creamos el MLP con `hidden_layer_sizes=(32, 16)` y la red aprendió automáticamente a separar medialunas. Nunca le dijimos "detecta curvas" — lo descubrió minimizando el loss.

El Playground simplemente **visualiza** lo que sklearn hace internamente.

### Replicar el Playground en sklearn

Veamos el equivalente del Experimento 4 (feature engineering con x₁² y x₂²):

In [ ]:
# Sin feature engineering: logística falla en círculos
lr_circ = LogisticRegression().fit(X_circles, y_circles)
acc_sin = accuracy_score(y_circles, lr_circ.predict(X_circles))

# CON feature engineering: agregamos x₁² y x₂²
X_circles_fe = np.column_stack([
    X_circles,
    X_circles[:, 0]**2,  # x₁²
    X_circles[:, 1]**2,  # x₂²
])
lr_circ_fe = LogisticRegression().fit(X_circles_fe, y_circles)
acc_con = accuracy_score(y_circles, lr_circ_fe.predict(X_circles_fe))

print(f"Logística SIN features extra: {acc_sin:.1%} (no puede)")
print(f"Logística CON x₁² y x₂²:     {acc_con:.1%} (¡ahora sí!)")
print()
print("¿Por qué funciona? Porque x₁² + x₂² = radio².")
print("Al darle el radio como feature, separar círculos es trivial.")
print()
print("La red neuronal hace esto INTERNAMENTE — cada neurona oculta")
print("crea su propia 'feature transformada' automáticamente.")
print("Por eso el MLP no necesita que hagamos feature engineering manual.")

---
## 10. Limitaciones del MLP y ¿Qué Viene Después?

El MLP es poderoso, pero tiene límites fundamentales con imágenes:

In [ ]:
# Demostración: el MLP no es invariante a posición
# Un '3' centrado vs un '3' desplazado son vectores COMPLETAMENTE diferentes

# Tomar un dígito y desplazarlo
sample = X_train[0].reshape(28, 28)
label = y_train[0]

from scipy.ndimage import shift

desplazamientos = [0, 3, 5, 7]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

for i, dx in enumerate(desplazamientos):
    shifted = shift(sample, [0, dx], mode="constant", cval=0)
    shifted_flat = shifted.flatten() / 255.0
    pred = mlp_mnist.predict(shifted_flat.reshape(1, -1))[0]
    proba = mlp_mnist.predict_proba(shifted_flat.reshape(1, -1))[0]
    conf = proba.max() * 100
    
    axes[i].imshow(shifted, cmap="gray_r")
    color = "#16A34A" if pred == label else "#C82B40"
    axes[i].set_title(f"Desplazado {dx}px\nPredicción: {pred} ({conf:.0f}%)",
                      fontweight="bold", fontsize=10, color=color)
    axes[i].axis("off")

plt.suptitle(f"Mismo dígito '{label}' desplazado — el MLP pierde precisión",
             fontweight="bold", fontsize=12)
plt.tight_layout()
plt.show()

### ¿Por qué falla?

El MLP **aplana** la imagen a un vector de 784 números. Pierde toda la estructura espacial:
- No sabe que el pixel 100 está **al lado** del pixel 101
- Un dígito desplazado 5 pixeles es un vector **completamente diferente**
- Tiene que aprender cada posición por separado → necesita muchísimos datos

### Las Redes Convolucionales (CNN) resuelven esto

| Limitación del MLP | Cómo lo resuelve la CNN |
|---|---|
| Aplana la imagen (pierde estructura 2D) | Procesa la imagen **como imagen** (2D) |
| No es invariante a posición | Usa **filtros locales** que recorren toda la imagen |
| Demasiados parámetros (784 × 128 = 100K) | Un filtro 3×3 tiene solo 9 pesos |
| No escala a imágenes reales (224×224×3) | Las CNNs procesan millones de pixeles eficientemente |

**Lo que aprendieron hoy (pesos, activaciones, backprop, curva de loss, early stopping) se reutiliza todo.** Solo cambia la arquitectura — de fully connected a convolucional.

Eso es lo que veremos en las próximas clases.

---
## Resumen

| Concepto | Detalle |
|---|---|
| **Logística = 1 neurona** | Misma operación: inputs × pesos + bias → activación |
| **Red neuronal** | Muchas neuronas en capas. La combinación crea curvas |
| **ReLU** | Activación estándar en capas ocultas. max(0, z) |
| **Learning rate** | Tamaño del paso. 0.001 default. Graficar curva de loss siempre |
| **MLPClassifier** | Misma API de sklearn. `hidden_layer_sizes`, `early_stopping=True` |
| **StandardScaler** | Obligatorio para redes neuronales |
| **Forma piramidal** | 128 → 64 → 32. Comprimir gradualmente |
| **TF Playground** | Las neuronas aprenden solas. Tú controlas estructura, no contenido |
| **Limitación MLP** | Aplana la imagen, no es invariante a posición → CNNs |
| **Flujo recomendado** | Logística → RF → MLP (solo si RF no alcanza) |